In [0]:
%run ../config

## Configuration file

Please change your catalog and schema here to run the demo on a different catalog.

<!-- Collect usage data (view). Remove it to disable collection or disable tracker during installation. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=2162748966026566&notebook=%2F_resources%2F00-setup&demo_name=lakehouse-hls-readmission&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-hls-readmission%2F_resources%2F00-setup&version=1">

In [0]:
dbutils.widgets.dropdown("reset_all_data", "false", ["true", "false"], "Reset all data")
reset_all_data = dbutils.widgets.get("reset_all_data") == "true"

In [0]:
%run ./00-global-setup-v2


# Technical Setup notebook. Hide this cell results
Initialize dataset to the current user and cleanup data when reset_all_data is set to true

Do not edit

In [0]:
DBDemos.setup_schema(catalog, db, reset_all_data, volume_name)
volume_folder =  f"/Volumes/{catalog}/{db}/{volume_name}"

USE CATALOG `main__build`
using catalog.database `main__build`.`dbdemos_hls_readmission`


In [0]:
#This  will download the data from our github repo to accelerate the demo start.
#Alternatively, you can run [00-generate-synthea-data]($./00-generate-synthea-data) to generate the data yourself with synthea.

reset_all_data = dbutils.widgets.get("reset_all_data") == "true"
import os
import requests
import timeit
import time
from datetime import datetime


folders = ["/landing_zone/encounters", "/landing_zone/patients", "/landing_zone/conditions", "/landing_zone/medications", "/landing_zone/immunizations", "/landing_zone/location_ref", "/landing_vocab/CONCEPT", "/landing_vocab/CONCEPT_RELATIONSHIP"]

if reset_all_data or DBDemos.is_any_folder_empty([volume_folder+f for f in folders]):
  if reset_all_data:
    assert len(volume_folder) > 20 and volume_folder.startswith('/Volumes/')
    dbutils.fs.rm(volume_folder, True)
  for f in folders:
      DBDemos.download_file_from_git(volume_folder+f, "databricks-demos", "dbdemos-dataset", "/hls/synthea"+f.replace("landing_zone", "landing_zone_parquet"))
else:
  print("data already existing. Run with reset_all_data=true to force a data cleanup for your local demo.")

data already existing. Run with reset_all_data=true to force a data cleanup for your local demo.


In [0]:
import time 
import plotly.express as px
import pandas as pd
import numpy as np

import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from pyspark.sql.types import *
from pyspark.sql.functions import *
import pyspark.sql.functions as F

from datetime import date

def drop_fs_table(table_name):
  try:
    fs.drop_table(table_name)  
  except Exception as e:
    print(f"Can't drop the fs table, probably doesn't exist? {e}")
  try:
    spark.sql(f"DROP TABLE IF EXISTS `{table_name}`")
  except Exception as e:
    print(f"Can't drop the delta table, probably doesn't exist? {e}")
